In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
recipes_path = "../../datasets/RAW_recipes.csv"

recipes = pd.read_csv(recipes_path)

print("Recipes dataset successfully loaded.")

In [ ]:
interactions_path = "../../datasets/RAW_interactions.csv"

interactions = pd.read_csv(interactions_path)

print("Interactions dataset successfully loaded.")

In [ ]:
print("Recipes shape:", recipes.shape)
print("Interactions shape:", interactions.shape)

print(interactions.head())
print(recipes.head())

print(interactions.columns.tolist())
print(recipes.columns.tolist())

In [ ]:
interactions["rating"].value_counts().sort_index()

In [ ]:
interactions["rating"].describe()

In [ ]:
import matplotlib.pyplot as plt

interactions["rating"].value_counts().sort_index().plot(
    kind="bar"
)

plt.xlabel("Rating")
plt.ylabel("Number of ratings")
plt.title("Rating distribution")
plt.show()

In [ ]:
duplicates = interactions.duplicated(
    subset=["user_id", "recipe_id"]
).sum()

print("Duplicate user-recipe pairs:", duplicates)

In [ ]:
user_counts = interactions["user_id"].value_counts()

print(user_counts.describe())

In [ ]:
recipe_counts = interactions["recipe_id"].value_counts()

print(recipe_counts.describe())

In [ ]:
print("Users with only 1 rating:", (user_counts == 1).sum())
print("Users with 5 or fewer ratings:", (user_counts <= 5).sum())

print("Recipes with only 1 rating:", (recipe_counts == 1).sum())
print("Recipes with 5 or fewer ratings:", (recipe_counts <= 5).sum())

In [ ]:
recipes["id"]

In [ ]:
interactions["recipe_id"]

In [ ]:
recipe_ids = set(recipes["id"])

missing_recipes = (
    ~interactions["recipe_id"].isin(recipe_ids)
).sum()

print("Interactions with missing recipes:", missing_recipes)

In [ ]:
data = interactions[
    ["user_id", "recipe_id", "rating"]
].copy()

print(data.head())
print(data.shape)

In [ ]:
data = data[data["rating"] > 0].copy()

print(data["rating"].value_counts().sort_index())

In [ ]:
user_ids = data["user_id"].unique()
recipe_ids = data["recipe_id"].unique()

user_to_idx = {
    user_id: idx
    for idx, user_id in enumerate(user_ids)
}

recipe_to_idx = {
    recipe_id: idx
    for idx, recipe_id in enumerate(recipe_ids)
}

In [ ]:
data["user_idx"] = data["user_id"].map(user_to_idx)
data["recipe_idx"] = data["recipe_id"].map(recipe_to_idx)

In [ ]:
data.head()

In [ ]:
num_users = len(user_to_idx)
num_recipes = len(recipe_to_idx)

print("Number of users:", num_users)
print("Number of recipes:", num_recipes)
print("Number of interactions:", len(data))

In [ ]:
data = interactions[
    ["user_id", "recipe_id", "rating"]
].copy()

# Izbacujemo rating 0
data = data[data["rating"] > 0].copy()

print("Interactions after removing rating 0:", len(data))

In [ ]:
MIN_USER_RATINGS = 5
MIN_RECIPE_RATINGS = 5

while True:
    old_size = len(data)

    # Broj ratinga po korisniku
    user_counts = data["user_id"].value_counts()

    # Broj ratinga po receptu
    recipe_counts = data["recipe_id"].value_counts()

    # Zadrži samo korisnike sa >= 5 ratinga
    data = data[
        data["user_id"].isin(
            user_counts[user_counts >= MIN_USER_RATINGS].index
        )
    ]

    # Zadrži samo recepte sa >= 5 ratinga
    data = data[
        data["recipe_id"].isin(
            recipe_counts[recipe_counts >= MIN_RECIPE_RATINGS].index
        )
    ]

    # Ako se ništa više ne menja, završili smo
    if len(data) == old_size:
        break

print("Final interactions:", len(data))
print("Users:", data["user_id"].nunique())
print("Recipes:", data["recipe_id"].nunique())

In [ ]:
user_counts = data["user_id"].value_counts()
recipe_counts = data["recipe_id"].value_counts()

print("Minimum user ratings:", user_counts.min())
print("Minimum recipe ratings:", recipe_counts.min())

print("\nUsers:")
print(user_counts.describe())

print("\nRecipes:")
print(recipe_counts.describe())

In [ ]:
print("Users with < 5 ratings:",
      (user_counts < 5).sum())

print("Recipes with < 5 ratings:",
      (recipe_counts < 5).sum())

In [ ]:
print("Final interactions:", len(data))
print("Users:", data["user_id"].nunique())
print("Recipes:", data["recipe_id"].nunique())
print("Minimum user ratings:", user_counts.min())
print("Minimum recipe ratings:", recipe_counts.min())

In [ ]:
data = interactions[
    ["user_id", "recipe_id", "rating", "date"]
].copy()

data = data[data["rating"] > 0].copy()

print(data.head())

In [ ]:
data["date"] = pd.to_datetime(data["date"])

print(data["date"].min())
print(data["date"].max())

In [ ]:
MIN_USER_RATINGS = 5
MIN_RECIPE_RATINGS = 5

while True:
    old_size = len(data)

    user_counts = data["user_id"].value_counts()
    recipe_counts = data["recipe_id"].value_counts()

    data = data[
        data["user_id"].isin(
            user_counts[user_counts >= MIN_USER_RATINGS].index
        )
    ]

    data = data[
        data["recipe_id"].isin(
            recipe_counts[recipe_counts >= MIN_RECIPE_RATINGS].index
        )
    ]

    if len(data) == old_size:
        break

print("Interactions:", len(data))
print("Users:", data["user_id"].nunique())
print("Recipes:", data["recipe_id"].nunique())

In [ ]:
data = data.sort_values("date").reset_index(drop=True)

In [ ]:
data[["user_id", "recipe_id", "rating", "date"]].head(10)

In [ ]:
data[["user_id", "recipe_id", "rating", "date"]].tail(10)

In [ ]:
n = len(data)

train_end = int(n * 0.80)
val_end = int(n * 0.90)

train = data.iloc[:train_end].copy()
validation = data.iloc[train_end:val_end].copy()
test = data.iloc[val_end:].copy()

print("Train:", len(train))
print("Validation:", len(validation))
print("Test:", len(test))

In [ ]:
train_users = set(train["user_id"])
val_users = set(validation["user_id"])
test_users = set(test["user_id"])

print("Validation users unseen in train:",
      len(val_users - train_users))

print("Test users unseen in train:",
      len(test_users - train_users))

In [ ]:
train_recipes = set(train["recipe_id"])
val_recipes = set(validation["recipe_id"])
test_recipes = set(test["recipe_id"])

print("Validation recipes unseen in train:",
      len(val_recipes - train_recipes))

print("Test recipes unseen in train:",
      len(test_recipes - train_recipes))

In [ ]:
user_ids = train["user_id"].unique()
recipe_ids = train["recipe_id"].unique()

user_to_idx = {
    user_id: idx
    for idx, user_id in enumerate(user_ids)
}

recipe_to_idx = {
    recipe_id: idx
    for idx, recipe_id in enumerate(recipe_ids)
}

In [ ]:
train["user_idx"] = train["user_id"].map(user_to_idx)
train["recipe_idx"] = train["recipe_id"].map(recipe_to_idx)

validation["user_idx"] = validation["user_id"].map(user_to_idx)
validation["recipe_idx"] = validation["recipe_id"].map(recipe_to_idx)

test["user_idx"] = test["user_id"].map(user_to_idx)
test["recipe_idx"] = test["recipe_id"].map(recipe_to_idx)

In [ ]:
print("Missing train users:",
      train["user_idx"].isna().sum())

print("Missing validation users:",
      validation["user_idx"].isna().sum())

print("Missing test users:",
      test["user_idx"].isna().sum())

print("Missing train recipes:",
      train["recipe_idx"].isna().sum())

print("Missing validation recipes:",
      validation["recipe_idx"].isna().sum())

print("Missing test recipes:",
      test["recipe_idx"].isna().sum())

In [ ]:
print("Train:", len(train))
print("Validation:", len(validation))
print("Test:", len(test))

print("Validation users unseen in train:",
      len(val_users - train_users))

print("Test users unseen in train:",
      len(test_users - train_users))

print("Validation recipes unseen in train:",
      len(val_recipes - train_recipes))

print("Test recipes unseen in train:",
      len(test_recipes - train_recipes))

In [ ]:
data = interactions[
    ["user_id", "recipe_id", "rating", "date"]
].copy()

# Izbacujemo rating 0
data = data[data["rating"] > 0].copy()

# Datum
data["date"] = pd.to_datetime(data["date"])

# Sortiramo po korisniku i vremenu
data = data.sort_values(
    ["user_id", "date"]
).reset_index(drop=True)

print(data.shape)
print(data.head())

In [ ]:
MIN_USER_RATINGS = 5
MIN_RECIPE_RATINGS = 5

while True:
    old_size = len(data)

    user_counts = data["user_id"].value_counts()
    recipe_counts = data["recipe_id"].value_counts()

    data = data[
        data["user_id"].isin(
            user_counts[user_counts >= MIN_USER_RATINGS].index
        )
    ]

    data = data[
        data["recipe_id"].isin(
            recipe_counts[recipe_counts >= MIN_RECIPE_RATINGS].index
        )
    ]

    if len(data) == old_size:
        break

print("Interactions:", len(data))
print("Users:", data["user_id"].nunique())
print("Recipes:", data["recipe_id"].nunique())

In [ ]:
train_list = []
val_list = []
test_list = []

for user_id, user_data in data.groupby("user_id"):

    # User data je već sortiran po datumu
    user_data = user_data.sort_values("date")

    train_list.append(user_data.iloc[:-2])
    val_list.append(user_data.iloc[-2:-1])
    test_list.append(user_data.iloc[-1:])

In [ ]:
train = pd.concat(train_list).reset_index(drop=True)
validation = pd.concat(val_list).reset_index(drop=True)
test = pd.concat(test_list).reset_index(drop=True)

In [ ]:
print("Train:", len(train))
print("Validation:", len(validation))
print("Test:", len(test))

In [ ]:
train_users = set(train["user_id"])
val_users = set(validation["user_id"])
test_users = set(test["user_id"])

print(
    "Validation users unseen in train:",
    len(val_users - train_users)
)

print(
    "Test users unseen in train:",
    len(test_users - train_users)
)

In [ ]:
train_recipes = set(train["recipe_id"])

val_unseen_recipes = (
    ~validation["recipe_id"].isin(train_recipes)
).sum()

test_unseen_recipes = (
    ~test["recipe_id"].isin(train_recipes)
).sum()

print(
    "Validation interactions with unseen recipes:",
    val_unseen_recipes
)

print(
    "Test interactions with unseen recipes:",
    test_unseen_recipes
)

In [ ]:
validation = validation[
    validation["recipe_id"].isin(train_recipes)
].copy()

test = test[
    test["recipe_id"].isin(train_recipes)
].copy()

In [ ]:
print("Train:", len(train))
print("Validation:", len(validation))
print("Test:", len(test))

In [ ]:
user_ids = train["user_id"].unique()
recipe_ids = train["recipe_id"].unique()

user_to_idx = {
    user_id: idx
    for idx, user_id in enumerate(user_ids)
}

recipe_to_idx = {
    recipe_id: idx
    for idx, recipe_id in enumerate(recipe_ids)
}

In [ ]:
num_users = len(user_to_idx)
num_recipes = len(recipe_to_idx)

print("Users:", num_users)
print("Recipes:", num_recipes)

In [ ]:
train["user_idx"] = train["user_id"].map(user_to_idx)
train["recipe_idx"] = train["recipe_id"].map(recipe_to_idx)

validation["user_idx"] = validation["user_id"].map(user_to_idx)
validation["recipe_idx"] = validation["recipe_id"].map(recipe_to_idx)

test["user_idx"] = test["user_id"].map(user_to_idx)
test["recipe_idx"] = test["recipe_id"].map(recipe_to_idx)

In [ ]:
print("Train missing:")
print(train[["user_idx", "recipe_idx"]].isna().sum())

print("\nValidation missing:")
print(validation[["user_idx", "recipe_idx"]].isna().sum())

print("\nTest missing:")
print(test[["user_idx", "recipe_idx"]].isna().sum())

In [ ]:
print("Train:", len(train))
print("Validation:", len(validation))
print("Test:", len(test))

print(
    "Validation users unseen:",
    len(val_users - train_users)
)

print(
    "Test users unseen:",
    len(test_users - train_users)
)

print(
    "Validation unseen recipes:",
    val_unseen_recipes
)

print(
    "Test unseen recipes:",
    test_unseen_recipes
)

In [ ]:
print("Users:", num_users)
print("Recipes:", num_recipes)

print(train[["user_id", "recipe_id", "rating", "user_idx", "recipe_idx"]].head())

In [ ]:
import numpy as np

# TRAIN
train_users = train["user_idx"].values.astype(np.int32)
train_recipes = train["recipe_idx"].values.astype(np.int32)
train_ratings = train["rating"].values.astype(np.float32)

# VALIDATION
val_users = validation["user_idx"].values.astype(np.int32)
val_recipes = validation["recipe_idx"].values.astype(np.int32)
val_ratings = validation["rating"].values.astype(np.float32)

# TEST
test_users = test["user_idx"].values.astype(np.int32)
test_recipes = test["recipe_idx"].values.astype(np.int32)
test_ratings = test["rating"].values.astype(np.float32)

print("Train users:", train_users.shape)
print("Train recipes:", train_recipes.shape)
print("Train ratings:", train_ratings.shape)

print("Validation:", val_users.shape)
print("Test:", test_users.shape)

In [ ]:
print("User:", train_users[0])
print("Recipe:", train_recipes[0])
print("Rating:", train_ratings[0])

In [ ]:
%pip install tensorflow

In [ ]:
import tensorflow as tf

print(tf.__version__)

In [ ]:
train_ds = tf.data.Dataset.from_tensor_slices(
    (
        {
            "user": train_users,
            "recipe": train_recipes
        },
        train_ratings
    )
)

val_ds = tf.data.Dataset.from_tensor_slices(
    (
        {
            "user": val_users,
            "recipe": val_recipes
        },
        val_ratings
    )
)

test_ds = tf.data.Dataset.from_tensor_slices(
    (
        {
            "user": test_users,
            "recipe": test_recipes
        },
        test_ratings
    )
)

In [ ]:
BATCH_SIZE = 512

train_ds = (
    train_ds
    .shuffle(100_000)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    val_ds
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

test_ds = (
    test_ds
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

In [ ]:
EMBEDDING_DIM = 64

user_input = keras.Input(
    shape=(),
    dtype=tf.int32,
    name="user"
)

recipe_input = keras.Input(
    shape=(),
    dtype=tf.int32,
    name="recipe"
)

In [ ]:
user_embedding = layers.Embedding(
    input_dim=num_users,
    output_dim=EMBEDDING_DIM,
    name="user_embedding"
)(user_input)

In [ ]:
recipe_embedding = layers.Embedding(
    input_dim=num_recipes,
    output_dim=EMBEDDING_DIM,
    name="recipe_embedding"
)(recipe_input)

In [ ]:
x = layers.Concatenate()(
    [user_embedding, recipe_embedding]
)

In [ ]:
x = layers.Dense(
    128,
    activation="relu"
)(x)

x = layers.Dropout(0.2)(x)

x = layers.Dense(
    64,
    activation="relu"
)(x)

x = layers.Dense(
    32,
    activation="relu"
)(x)

In [ ]:
output = layers.Dense(
    1,
    activation=None,
    name="rating"
)(x)

In [ ]:
model = keras.Model(
    inputs={
        "user": user_input,
        "recipe": recipe_input
    },
    outputs=output
)

In [ ]:
model.summary()

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=[
        keras.metrics.RootMeanSquaredError(name="rmse"),
        keras.metrics.MeanAbsoluteError(name="mae")
    ]
)

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True
)

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=[early_stopping]
)

In [ ]:
def build_ncf_model(
        num_users,
        num_recipes,
        embedding_dim=32,
        dropout_rate=0.3,
        learning_rate=0.001
):

    user_input = keras.Input(
        shape=(),
        dtype=tf.int32,
        name="user"
    )

    recipe_input = keras.Input(
        shape=(),
        dtype=tf.int32,
        name="recipe"
    )

    user_embedding = layers.Embedding(
        input_dim=num_users,
        output_dim=embedding_dim,
        name="user_embedding"
    )(user_input)

    recipe_embedding = layers.Embedding(
        input_dim=num_recipes,
        output_dim=embedding_dim,
        name="recipe_embedding"
    )(recipe_input)

    x = layers.Concatenate()(
        [user_embedding, recipe_embedding]
    )

    x = layers.Dense(
        64,
        activation="relu"
    )(x)

    x = layers.Dropout(
        dropout_rate
    )(x)

    x = layers.Dense(
        32,
        activation="relu"
    )(x)

    output = layers.Dense(
        1,
        name="rating"
    )(x)

    model = keras.Model(
        inputs={
            "user": user_input,
            "recipe": recipe_input
        },
        outputs=output
    )

    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=learning_rate
        ),
        loss="mse",
        metrics=[
            keras.metrics.RootMeanSquaredError(name="rmse"),
            keras.metrics.MeanAbsoluteError(name="mae")
        ]
    )

    return model

In [ ]:
model_v2 = build_ncf_model(
    num_users=num_users,
    num_recipes=num_recipes,
    embedding_dim=32,
    dropout_rate=0.3,
    learning_rate=0.001
)

In [ ]:
model_v2.summary()

In [ ]:
early_stopping_v2 = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=6,
    restore_best_weights=True
)

history_v2 = model_v2.fit(
    train_ds,
    validation_data=val_ds,
    epochs=35,
    callbacks=[early_stopping_v2]
)

In [ ]:
test_results_v2 = model_v2.evaluate(
    test_ds,
    return_dict=True
)

print(test_results_v2)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

plt.plot(history_v2.history["loss"], label="Train Loss")
plt.plot(history_v2.history["val_loss"], label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("NCF V2 - Training vs Validation Loss")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(history_v2.history["rmse"], label="Train RMSE")
plt.plot(history_v2.history["val_rmse"], label="Validation RMSE")

plt.xlabel("Epoch")
plt.ylabel("RMSE")
plt.title("NCF V2 - Training vs Validation RMSE")
plt.legend()
plt.show()

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers, regularizers

In [ ]:
EMBEDDING_DIM = 32
L2_REG = 1e-6

user_input = keras.Input(shape=(1,), name="user")
recipe_input = keras.Input(shape=(1,), name="recipe")

user_embedding = layers.Embedding(
    input_dim=num_users,
    output_dim=EMBEDDING_DIM,
    embeddings_regularizer=regularizers.l2(L2_REG),
    name="user_embedding"
)(user_input)

recipe_embedding = layers.Embedding(
    input_dim=num_recipes,
    output_dim=EMBEDDING_DIM,
    embeddings_regularizer=regularizers.l2(L2_REG),
    name="recipe_embedding"
)(recipe_input)

user_vector = layers.Flatten()(user_embedding)
recipe_vector = layers.Flatten()(recipe_embedding)

x = layers.Concatenate()([user_vector, recipe_vector])

x = layers.Dense(
    64,
    activation="relu",
    kernel_regularizer=regularizers.l2(L2_REG)
)(x)

x = layers.Dropout(0.30)(x)

x = layers.Dense(
    32,
    activation="relu",
    kernel_regularizer=regularizers.l2(L2_REG)
)(x)

x = layers.Dropout(0.20)(x)

output = layers.Dense(1, activation="linear", name="rating")(x)

model_v3 = keras.Model(
    inputs=[user_input, recipe_input],
    outputs=output
)

model_v3.summary()

In [ ]:
optimizer_v3 = keras.optimizers.Adam(
    learning_rate=0.001
)

model_v3.compile(
    optimizer=optimizer_v3,
    loss="mse",
    metrics=[
        keras.metrics.MeanAbsoluteError(name="mae"),
        keras.metrics.RootMeanSquaredError(name="rmse")
    ]
)

In [ ]:
early_stopping_v3 = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

In [ ]:
history_v3 = model_v3.fit(
    train_ds,
    validation_data=val_ds,
    epochs=35,
    callbacks=[early_stopping_v3]
)

In [ ]:
test_results_v3 = model_v3.evaluate(
    test_ds,
    return_dict=True
)

print(test_results_v3)

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers, regularizers

In [ ]:
EMBEDDING_DIM = 32
L2_REG = 1e-6

# Inputs
user_input = keras.Input(shape=(1,), name="user")
recipe_input = keras.Input(shape=(1,), name="recipe")


# =========================
# User representation
# =========================

user_embedding = layers.Embedding(
    input_dim=num_users,
    output_dim=EMBEDDING_DIM,
    embeddings_regularizer=regularizers.l2(L2_REG),
    name="user_embedding"
)(user_input)

user_vector = layers.Flatten()(user_embedding)


# User bias
user_bias = layers.Embedding(
    input_dim=num_users,
    output_dim=1,
    embeddings_regularizer=regularizers.l2(L2_REG),
    name="user_bias"
)(user_input)

user_bias = layers.Flatten()(user_bias)


# =========================
# Recipe representation
# =========================

recipe_embedding = layers.Embedding(
    input_dim=num_recipes,
    output_dim=EMBEDDING_DIM,
    embeddings_regularizer=regularizers.l2(L2_REG),
    name="recipe_embedding"
)(recipe_input)

recipe_vector = layers.Flatten()(recipe_embedding)


# Recipe bias
recipe_bias = layers.Embedding(
    input_dim=num_recipes,
    output_dim=1,
    embeddings_regularizer=regularizers.l2(L2_REG),
    name="recipe_bias"
)(recipe_input)

recipe_bias = layers.Flatten()(recipe_bias)


# =========================
# NCF interaction
# =========================

interaction = layers.Concatenate()([
    user_vector,
    recipe_vector
])

x = layers.Dense(
    64,
    activation="relu",
    kernel_regularizer=regularizers.l2(L2_REG)
)(interaction)

x = layers.Dropout(0.20)(x)

x = layers.Dense(
    32,
    activation="relu",
    kernel_regularizer=regularizers.l2(L2_REG)
)(x)

x = layers.Dropout(0.10)(x)


# =========================
# Combine NCF + biases
# =========================

x = layers.Concatenate()([
    x,
    user_bias,
    recipe_bias
])

output = layers.Dense(
    1,
    activation="linear",
    name="rating"
)(x)


# =========================
# Model
# =========================

model_v4 = keras.Model(
    inputs=[user_input, recipe_input],
    outputs=output
)

model_v4.summary()

In [ ]:
model_v4.summary()

In [ ]:
optimizer_v4 = keras.optimizers.Adam(
    learning_rate=0.001
)

model_v4.compile(
    optimizer=optimizer_v4,
    loss="mse",
    metrics=[
        keras.metrics.MeanAbsoluteError(name="mae"),
        keras.metrics.RootMeanSquaredError(name="rmse")
    ]
)

print("V4 model compiled successfully.")

In [ ]:
early_stopping_v4 = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

In [ ]:
history_v4 = model_v4.fit(
    train_ds,
    validation_data=val_ds,
    epochs=35,
    callbacks=[early_stopping_v4]
)

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

# =========================
# NCF V5
# =========================

embedding_dim = 16

user_input = keras.Input(shape=(1,), name="user")
recipe_input = keras.Input(shape=(1,), name="recipe")

# User embedding
user_embedding = layers.Embedding(
    input_dim=num_users,
    output_dim=embedding_dim,
    embeddings_regularizer=regularizers.l2(1e-6),
    name="user_embedding"
)(user_input)

# Recipe embedding
recipe_embedding = layers.Embedding(
    input_dim=num_recipes,
    output_dim=embedding_dim,
    embeddings_regularizer=regularizers.l2(1e-6),
    name="recipe_embedding"
)(recipe_input)

user_vec = layers.Flatten()(user_embedding)
recipe_vec = layers.Flatten()(recipe_embedding)

# Combine user + recipe
x = layers.Concatenate()([user_vec, recipe_vec])

# Shallow MLP
x = layers.Dense(
    32,
    activation="relu",
    kernel_regularizer=regularizers.l2(1e-4)
)(x)

x = layers.Dropout(0.30)(x)

x = layers.Dense(
    16,
    activation="relu",
    kernel_regularizer=regularizers.l2(1e-4)
)(x)

x = layers.Dropout(0.20)(x)

# Rating prediction
rating_output = layers.Dense(
    1,
    activation="linear",
    name="rating"
)(x)

model_v5 = keras.Model(
    inputs=[user_input, recipe_input],
    outputs=rating_output
)

model_v5.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=[
        keras.metrics.MeanAbsoluteError(name="mae"),
        keras.metrics.RootMeanSquaredError(name="rmse")
    ]
)

model_v5.summary()

In [ ]:
early_stopping_v5 = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    min_delta=0.001,
    restore_best_weights=True
)

In [ ]:
history_v5 = model_v5.fit(
    train_ds,
    validation_data=val_ds,
    epochs=35,
    callbacks=[early_stopping_v5]
)

In [ ]:
v5_results = model_v5.evaluate(
    test_ds,
    return_dict=True
)

print(v5_results)

In [ ]:
# ==========================================
# NCF V6 - NeuMF
# ==========================================

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

embedding_dim = 16

# Inputs
user_input = keras.Input(shape=(1,), name="user")
recipe_input = keras.Input(shape=(1,), name="recipe")


# ------------------------------------------
# GMF branch
# ------------------------------------------

gmf_user_embedding = layers.Embedding(
    input_dim=num_users,
    output_dim=embedding_dim,
    name="gmf_user_embedding"
)(user_input)

gmf_recipe_embedding = layers.Embedding(
    input_dim=num_recipes,
    output_dim=embedding_dim,
    name="gmf_recipe_embedding"
)(recipe_input)

gmf_user = layers.Flatten()(gmf_user_embedding)
gmf_recipe = layers.Flatten()(gmf_recipe_embedding)

# Element-wise multiplication
gmf = layers.Multiply()([gmf_user, gmf_recipe])


# ------------------------------------------
# MLP branch
# ------------------------------------------

mlp_user_embedding = layers.Embedding(
    input_dim=num_users,
    output_dim=embedding_dim,
    embeddings_regularizer=regularizers.l2(1e-6),
    name="mlp_user_embedding"
)(user_input)

mlp_recipe_embedding = layers.Embedding(
    input_dim=num_recipes,
    output_dim=embedding_dim,
    embeddings_regularizer=regularizers.l2(1e-6),
    name="mlp_recipe_embedding"
)(recipe_input)

mlp_user = layers.Flatten()(mlp_user_embedding)
mlp_recipe = layers.Flatten()(mlp_recipe_embedding)

mlp = layers.Concatenate()([mlp_user, mlp_recipe])

mlp = layers.Dense(
    32,
    activation="relu",
    kernel_regularizer=regularizers.l2(1e-4)
)(mlp)

mlp = layers.Dropout(0.3)(mlp)

mlp = layers.Dense(
    16,
    activation="relu",
    kernel_regularizer=regularizers.l2(1e-4)
)(mlp)

mlp = layers.Dropout(0.2)(mlp)


# ------------------------------------------
# Combine GMF + MLP
# ------------------------------------------

combined = layers.Concatenate()([gmf, mlp])

rating_output = layers.Dense(
    1,
    activation="linear",
    name="rating"
)(combined)


# ------------------------------------------
# Build model
# ------------------------------------------

model_v6 = keras.Model(
    inputs=[user_input, recipe_input],
    outputs=rating_output
)


# ------------------------------------------
# Compile
# ------------------------------------------

model_v6.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=[
        keras.metrics.MeanAbsoluteError(name="mae"),
        keras.metrics.RootMeanSquaredError(name="rmse")
    ]
)

model_v6.summary()

In [ ]:
early_stopping_v6 = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=10,
    min_delta=0.001,
    restore_best_weights=True
)

In [ ]:
history_v6 = model_v6.fit(
    train_ds,
    validation_data=val_ds,
    epochs=55,
    callbacks=[early_stopping_v6]
)

In [ ]:
global_mean = train["rating"].mean()

pred_global = np.full(
    len(test),
    global_mean
)

global_rmse = np.sqrt(
    np.mean(
        (test["rating"].values - pred_global) ** 2
    )
)

global_mae = np.mean(
    np.abs(
        test["rating"].values - pred_global
    )
)

print("Global mean:", global_mean)
print("Global RMSE:", global_rmse)
print("Global MAE:", global_mae)

In [ ]:
user_means = (
    train.groupby("user_id")["rating"]
    .mean()
)

test_user_mean = test["user_id"].map(user_means)

# Ako se nekim slučajem pojavi user koji nema train rating,
# koristimo globalni prosek
test_user_mean = test_user_mean.fillna(global_mean)

user_rmse = np.sqrt(
    np.mean(
        (test["rating"].values - test_user_mean.values) ** 2
    )
)

user_mae = np.mean(
    np.abs(
        test["rating"].values - test_user_mean.values
    )
)

print("User mean RMSE:", user_rmse)
print("User mean MAE:", user_mae)

In [ ]:
recipe_means = (
    train.groupby("recipe_id")["rating"]
    .mean()
)

test_recipe_mean = test["recipe_id"].map(recipe_means)

test_recipe_mean = test_recipe_mean.fillna(global_mean)

user_means_test = (
    test["user_id"]
    .map(user_means)
    .fillna(global_mean)
)

# Kombinujemo user i recipe preference
pred_bias = (
        user_means_test
        + test_recipe_mean
        - global_mean
)

# Ograničimo predikciju na validan rating range
pred_bias = pred_bias.clip(0, 5)

bias_rmse = np.sqrt(
    np.mean(
        (test["rating"].values - pred_bias.values) ** 2
    )
)

bias_mae = np.mean(
    np.abs(
        test["rating"].values - pred_bias.values
    )
)

print("User + Recipe bias RMSE:", bias_rmse)
print("User + Recipe bias MAE:", bias_mae)

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

tf.random.set_seed(42)
np.random.seed(42)

embedding_dim = 32

user_input = keras.Input(shape=(1,), name="user")
recipe_input = keras.Input(shape=(1,), name="recipe")

user_embedding = layers.Embedding(
    input_dim=num_users,
    output_dim=embedding_dim,
    embeddings_regularizer=regularizers.l2(1e-6),
    name="user_embedding"
)(user_input)

recipe_embedding = layers.Embedding(
    input_dim=num_recipes,
    output_dim=embedding_dim,
    embeddings_regularizer=regularizers.l2(1e-6),
    name="recipe_embedding"
)(recipe_input)

user_vector = layers.Flatten()(user_embedding)
recipe_vector = layers.Flatten()(recipe_embedding)

x = layers.Concatenate()([user_vector, recipe_vector])

x = layers.Dense(
    32,
    activation="relu",
    kernel_regularizer=regularizers.l2(1e-4)
)(x)

x = layers.Dropout(0.30)(x)

output = layers.Dense(1, name="rating")(x)

model_v6 = keras.Model(
    inputs=[user_input, recipe_input],
    outputs=output,
    name="NCF_V6"
)

model_v6.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="mse",
    metrics=[
        "mae",
        keras.metrics.RootMeanSquaredError(name="rmse")
    ]
)

model_v6.summary()

In [ ]:
early_stopping_v6 = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=7,
    min_delta=0.001,
    restore_best_weights=True
)

In [ ]:
history_v6 = model_v6.fit(
    train_ds,
    validation_data=val_ds,
    epochs=35,
    callbacks=[early_stopping_v6]
)

In [ ]:
v6_results = model_v6.evaluate(
    test_ds,
    return_dict=True
)

print(v6_results)

In [ ]:
MIN_USER_RATINGS = 10
MIN_RECIPE_RATINGS = 10

filtered_ratings_v7 = explicit_ratings.copy()

while True:
    user_counts = filtered_ratings_v7.groupby("user_id").size()
    recipe_counts = filtered_ratings_v7.groupby("recipe_id").size()

    valid_users = user_counts[user_counts >= MIN_USER_RATINGS].index
    valid_recipes = recipe_counts[recipe_counts >= MIN_RECIPE_RATINGS].index

    new_filtered = filtered_ratings_v7[
        filtered_ratings_v7["user_id"].isin(valid_users)
        & filtered_ratings_v7["recipe_id"].isin(valid_recipes)
        ].copy()

    if len(new_filtered) == len(filtered_ratings_v7):
        break

    filtered_ratings_v7 = new_filtered

print("Interactions:", len(filtered_ratings_v7))
print("Users:", filtered_ratings_v7["user_id"].nunique())
print("Recipes:", filtered_ratings_v7["recipe_id"].nunique())

In [ ]:
MIN_USER_RATINGS = 10
MIN_RECIPE_RATINGS = 10

filtered_ratings_v7 = interactions[
    interactions["rating"] > 0
    ].copy()

while True:
    user_counts = filtered_ratings_v7.groupby("user_id").size()
    recipe_counts = filtered_ratings_v7.groupby("recipe_id").size()

    valid_users = user_counts[
        user_counts >= MIN_USER_RATINGS
        ].index

    valid_recipes = recipe_counts[
        recipe_counts >= MIN_RECIPE_RATINGS
        ].index

    new_filtered = filtered_ratings_v7[
        filtered_ratings_v7["user_id"].isin(valid_users)
        & filtered_ratings_v7["recipe_id"].isin(valid_recipes)
        ].copy()

    if len(new_filtered) == len(filtered_ratings_v7):
        break

    filtered_ratings_v7 = new_filtered

print("V7 interactions:", len(filtered_ratings_v7))
print("V7 users:", filtered_ratings_v7["user_id"].nunique())
print("V7 recipes:", filtered_ratings_v7["recipe_id"].nunique())

In [ ]:
explicit_ratings = interactions[interactions["rating"] > 0].copy()

In [ ]:
from sklearn.model_selection import train_test_split

train_ratings_v7, test_ratings_v7 = train_test_split(
    filtered_ratings_v7,
    test_size=0.20,
    random_state=42
)

print("Train interactions:", len(train_ratings_v7))
print("Test interactions:", len(test_ratings_v7))
print("Total:", len(train_ratings_v7) + len(test_ratings_v7))

In [ ]:
train_users_v7 = set(train_ratings_v7["user_id"])
test_users_v7 = set(test_ratings_v7["user_id"])

train_recipes_v7 = set(train_ratings_v7["recipe_id"])
test_recipes_v7 = set(test_ratings_v7["recipe_id"])

cold_start_users_v7 = test_users_v7 - train_users_v7
cold_start_recipes_v7 = test_recipes_v7 - train_recipes_v7

print("Cold-start users:", len(cold_start_users_v7))
print("Cold-start recipes:", len(cold_start_recipes_v7))

In [ ]:
# V7 - Encode users and recipes

user_ids_v7 = train_ratings_v7["user_id"].unique()
recipe_ids_v7 = train_ratings_v7["recipe_id"].unique()

user_to_index_v7 = {
    user_id: idx
    for idx, user_id in enumerate(user_ids_v7)
}

recipe_to_index_v7 = {
    recipe_id: idx
    for idx, recipe_id in enumerate(recipe_ids_v7)
}

train_ratings_v7 = train_ratings_v7.copy()
test_ratings_v7 = test_ratings_v7.copy()

train_ratings_v7["user_idx"] = train_ratings_v7["user_id"].map(user_to_index_v7)
train_ratings_v7["recipe_idx"] = train_ratings_v7["recipe_id"].map(recipe_to_index_v7)

test_ratings_v7["user_idx"] = test_ratings_v7["user_id"].map(user_to_index_v7)
test_ratings_v7["recipe_idx"] = test_ratings_v7["recipe_id"].map(recipe_to_index_v7)

print("V7 users:", len(user_to_index_v7))
print("V7 recipes:", len(recipe_to_index_v7))

In [ ]:
# ============================================
# V7 - Mapiranje user/recipe ID-eva
# ============================================

user_ids = train_ratings_v7["user_id"].unique()
recipe_ids = train_ratings_v7["recipe_id"].unique()

user2idx_v7 = {user_id: idx for idx, user_id in enumerate(user_ids)}
recipe2idx_v7 = {recipe_id: idx for idx, recipe_id in enumerate(recipe_ids)}

train_v7 = train_ratings_v7.copy()
test_v7 = test_ratings_v7.copy()

train_v7["user_idx"] = train_v7["user_id"].map(user2idx_v7)
train_v7["recipe_idx"] = train_v7["recipe_id"].map(recipe2idx_v7)

test_v7["user_idx"] = test_v7["user_id"].map(user2idx_v7)
test_v7["recipe_idx"] = test_v7["recipe_id"].map(recipe2idx_v7)

print("Users:", len(user2idx_v7))
print("Recipes:", len(recipe2idx_v7))

print("Missing user mappings:", train_v7["user_idx"].isna().sum())
print("Missing recipe mappings:", train_v7["recipe_idx"].isna().sum())

print("Test missing user mappings:", test_v7["user_idx"].isna().sum())
print("Test missing recipe mappings:", test_v7["recipe_idx"].isna().sum())

In [ ]:
import tensorflow as tf

BATCH_SIZE = 256

X_train_v7 = {
    "user": train_v7["user_idx"].astype("int32").values,
    "recipe": train_v7["recipe_idx"].astype("int32").values
}

y_train_v7 = train_v7["rating"].astype("float32").values

X_test_v7 = {
    "user": test_v7["user_idx"].astype("int32").values,
    "recipe": test_v7["recipe_idx"].astype("int32").values
}

y_test_v7 = test_v7["rating"].astype("float32").values

train_ds_v7 = tf.data.Dataset.from_tensor_slices(
    (X_train_v7, y_train_v7)
).shuffle(
    10000,
    seed=42
).batch(
    BATCH_SIZE
).prefetch(
    tf.data.AUTOTUNE
)

test_ds_v7 = tf.data.Dataset.from_tensor_slices(
    (X_test_v7, y_test_v7)
).batch(
    BATCH_SIZE
).prefetch(
    tf.data.AUTOTUNE
)

print("Train:", len(train_v7))
print("Test:", len(test_v7))

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers, regularizers

NUM_USERS_V7 = len(user2idx_v7)
NUM_RECIPES_V7 = len(recipe2idx_v7)

EMBEDDING_DIM_V7 = 16

user_input_v7 = keras.Input(
    shape=(1,),
    name="user"
)

recipe_input_v7 = keras.Input(
    shape=(1,),
    name="recipe"
)

user_embedding_v7 = layers.Embedding(
    input_dim=NUM_USERS_V7,
    output_dim=EMBEDDING_DIM_V7,
    embeddings_regularizer=regularizers.l2(1e-5),
    name="user_embedding"
)(user_input_v7)

recipe_embedding_v7 = layers.Embedding(
    input_dim=NUM_RECIPES_V7,
    output_dim=EMBEDDING_DIM_V7,
    embeddings_regularizer=regularizers.l2(1e-5),
    name="recipe_embedding"
)(recipe_input_v7)

user_vector_v7 = layers.Flatten()(user_embedding_v7)
recipe_vector_v7 = layers.Flatten()(recipe_embedding_v7)

concat_v7 = layers.Concatenate()([
    user_vector_v7,
    recipe_vector_v7
])

dense_v7 = layers.Dense(
    32,
    activation="relu",
    kernel_regularizer=regularizers.l2(1e-4)
)(concat_v7)

dropout_v7 = layers.Dropout(0.30)(dense_v7)

dense2_v7 = layers.Dense(
    16,
    activation="relu",
    kernel_regularizer=regularizers.l2(1e-4)
)(dropout_v7)

dropout2_v7 = layers.Dropout(0.20)(dense2_v7)

rating_output_v7 = layers.Dense(
    1,
    activation="linear",
    name="rating"
)(dropout2_v7)

model_v7 = keras.Model(
    inputs=[user_input_v7, recipe_input_v7],
    outputs=rating_output_v7
)

model_v7.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=[
        keras.metrics.MeanAbsoluteError(name="mae"),
        keras.metrics.RootMeanSquaredError(name="rmse")
    ]
)

model_v7.summary()

In [ ]:
early_stopping_v7 = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=10,
    min_delta=0.001,
    restore_best_weights=True
)

In [ ]:
history_v7 = model_v7.fit(
    train_ds_v7,
    validation_data=test_ds_v7,
    epochs=55,
    callbacks=[early_stopping_v7]
)

In [ ]:
from sklearn.model_selection import train_test_split

train_mf, val_mf = train_test_split(
    train_ratings_v7,
    test_size=0.10,
    random_state=42
)

print("MF train:", len(train_mf))
print("MF validation:", len(val_mf))
print("Final test:", len(test_ratings_v7))

In [ ]:
BATCH_SIZE_MF = 256

X_train_mf = {
    "user": train_mf["user_id"].map(user2idx_v7).astype("int32").values,
    "recipe": train_mf["recipe_id"].map(recipe2idx_v7).astype("int32").values
}

y_train_mf = train_mf["rating"].astype("float32").values


X_val_mf = {
    "user": val_mf["user_id"].map(user2idx_v7).astype("int32").values,
    "recipe": val_mf["recipe_id"].map(recipe2idx_v7).astype("int32").values
}

y_val_mf = val_mf["rating"].astype("float32").values


X_test_mf = {
    "user": test_ratings_v7["user_id"].map(user2idx_v7).astype("int32").values,
    "recipe": test_ratings_v7["recipe_id"].map(recipe2idx_v7).astype("int32").values
}

y_test_mf = test_ratings_v7["rating"].astype("float32").values


train_ds_mf = tf.data.Dataset.from_tensor_slices(
    (X_train_mf, y_train_mf)
).shuffle(
    10000,
    seed=42
).batch(
    BATCH_SIZE_MF
).prefetch(
    tf.data.AUTOTUNE
)


val_ds_mf = tf.data.Dataset.from_tensor_slices(
    (X_val_mf, y_val_mf)
).batch(
    BATCH_SIZE_MF
).prefetch(
    tf.data.AUTOTUNE
)


test_ds_mf = tf.data.Dataset.from_tensor_slices(
    (X_test_mf, y_test_mf)
).batch(
    BATCH_SIZE_MF
).prefetch(
    tf.data.AUTOTUNE
)

print("Datasets ready.")

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers, regularizers

EMBEDDING_DIM_MF = 32

user_input_mf = keras.Input(
    shape=(1,),
    dtype="int32",
    name="user"
)

recipe_input_mf = keras.Input(
    shape=(1,),
    dtype="int32",
    name="recipe"
)


user_embedding_mf = layers.Embedding(
    input_dim=NUM_USERS_V7,
    output_dim=EMBEDDING_DIM_MF,
    embeddings_regularizer=regularizers.l2(1e-6),
    name="user_embedding_mf"
)(user_input_mf)

recipe_embedding_mf = layers.Embedding(
    input_dim=NUM_RECIPES_V7,
    output_dim=EMBEDDING_DIM_MF,
    embeddings_regularizer=regularizers.l2(1e-6),
    name="recipe_embedding_mf"
)(recipe_input_mf)


user_bias_mf = layers.Embedding(
    input_dim=NUM_USERS_V7,
    output_dim=1,
    embeddings_regularizer=regularizers.l2(1e-6),
    name="user_bias_mf"
)(user_input_mf)

recipe_bias_mf = layers.Embedding(
    input_dim=NUM_RECIPES_V7,
    output_dim=1,
    embeddings_regularizer=regularizers.l2(1e-6),
    name="recipe_bias_mf"
)(recipe_input_mf)


user_vector_mf = layers.Flatten()(user_embedding_mf)
recipe_vector_mf = layers.Flatten()(recipe_embedding_mf)

user_bias_mf = layers.Flatten()(user_bias_mf)
recipe_bias_mf = layers.Flatten()(recipe_bias_mf)


dot_product_mf = layers.Dot(
    axes=1
)([
    user_vector_mf,
    recipe_vector_mf
])


global_mean_mf = float(train_mf["rating"].mean())

global_mean_layer = layers.Lambda(
    lambda x: x * 0.0 + global_mean_mf,
    name="global_mean"
)(dot_product_mf)


rating_output_mf = layers.Add(name="rating")([
    dot_product_mf,
    user_bias_mf,
    recipe_bias_mf,
    global_mean_layer
])


model_v8 = keras.Model(
    inputs=[user_input_mf, recipe_input_mf],
    outputs=rating_output_mf
)


model_v8.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=[
        keras.metrics.MeanAbsoluteError(name="mae"),
        keras.metrics.RootMeanSquaredError(name="rmse")
    ]
)

model_v8.summary()

In [ ]:
early_stopping_v8 = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=7,
    min_delta=0.001,
    restore_best_weights=True
)

In [ ]:
history_v8 = model_v8.fit(
    train_ds_mf,
    validation_data=val_ds_mf,
    epochs=40,
    callbacks=[early_stopping_v8]
)

In [ ]:
v8_test_results = model_v8.evaluate(
    test_ds_mf,
    return_dict=True
)

print(v8_test_results)

In [ ]:
print(model_v7)
print(len(user2idx_v7))
print(len(recipe2idx_v7))

In [ ]:
# ============================================
# V7 - Precision@10 and Recall@10
# ============================================

K = 10
RELEVANT_THRESHOLD = 4

# Svi korisnici koji imaju test interakcije
eval_users = test_ratings_v7["user_id"].unique()

# Relevantni recepti u test skupu: rating >= 4
test_relevant = (
    test_ratings_v7[test_ratings_v7["rating"] >= RELEVANT_THRESHOLD]
    .groupby("user_id")["recipe_id"]
    .apply(set)
    .to_dict()
)

# Recepti koje je svaki korisnik već imao u TRAIN skupu
train_seen = (
    train_ratings_v7
    .groupby("user_id")["recipe_id"]
    .apply(set)
    .to_dict()
)

# Samo korisnici koji imaju bar jedan relevantan recept u testu
eval_users = [
    user_id
    for user_id in eval_users
    if user_id in test_relevant and len(test_relevant[user_id]) > 0
]

print("Evaluation users:", len(eval_users))

# Svi recepti koje model poznaje
all_recipe_ids = np.array(list(recipe2idx_v7.keys()))
all_recipe_indices = np.array(
    [recipe2idx_v7[rid] for rid in all_recipe_ids],
    dtype=np.int32
)

user_indices = np.array(
    [user2idx_v7[uid] for uid in eval_users],
    dtype=np.int32
)

precision_scores = []
recall_scores = []

# Radimo više korisnika odjednom da ne trošimo ogromnu memoriju
USER_BATCH_SIZE = 64

for start in range(0, len(eval_users), USER_BATCH_SIZE):

    batch_users = eval_users[start:start + USER_BATCH_SIZE]
    batch_user_indices = user_indices[start:start + USER_BATCH_SIZE]

    n_users = len(batch_users)
    n_recipes = len(all_recipe_indices)

    # Svaki korisnik dobija sve recepte kao kandidate
    user_array = np.repeat(
        batch_user_indices,
        n_recipes
    )

    recipe_array = np.tile(
        all_recipe_indices,
        n_users
    )

    # Predikcije V7 modela
    predictions = model_v7.predict(
        {
            "user": user_array,
            "recipe": recipe_array
        },
        batch_size=4096,
        verbose=0
    ).reshape(n_users, n_recipes)

    # Za svakog korisnika uklanjamo recepte
    # koje je već imao u TRAIN skupu
    for i, user_id in enumerate(batch_users):

        seen_recipes = train_seen.get(user_id, set())

        for recipe_id in seen_recipes:
            if recipe_id in recipe2idx_v7:
                recipe_idx = recipe2idx_v7[recipe_id]
                recipe_position = np.where(
                    all_recipe_indices == recipe_idx
                )[0]

                if len(recipe_position) > 0:
                    predictions[i, recipe_position[0]] = -np.inf

        # Top 10 preporuka
        top10_positions = np.argpartition(
            predictions[i],
            -K
        )[-K:]

        top10_positions = top10_positions[
            np.argsort(predictions[i][top10_positions])[::-1]
        ]

        recommended_recipes = set(
            all_recipe_ids[top10_positions]
        )

        relevant_recipes = test_relevant[user_id]

        hits = len(
            recommended_recipes.intersection(relevant_recipes)
        )

        precision = hits / K
        recall = hits / len(relevant_recipes)

        precision_scores.append(precision)
        recall_scores.append(recall)

    print(
        f"Processed {min(start + USER_BATCH_SIZE, len(eval_users))}"
        f"/{len(eval_users)} users"
    )


precision_at_10 = np.mean(precision_scores)
recall_at_10 = np.mean(recall_scores)

print("\n====================================")
print("V7 Recommendation Evaluation")
print("====================================")
print(f"Precision@10: {precision_at_10:.4f}")
print(f"Recall@10:    {recall_at_10:.4f}")
print(f"Users evaluated: {len(eval_users)}")

In [ ]:
# Pozitivne interakcije
positive_train_v9 = train_ratings_v7[
    train_ratings_v7["rating"] >= 4
    ].copy()

positive_test_v9 = test_ratings_v7[
    test_ratings_v7["rating"] >= 4
    ].copy()

print("Positive train interactions:", len(positive_train_v9))
print("Positive test interactions:", len(positive_test_v9))
print("Train users:", positive_train_v9["user_id"].nunique())
print("Test users:", positive_test_v9["user_id"].nunique())

In [ ]:
user_ids_v9 = sorted(train_ratings_v7["user_id"].unique())
recipe_ids_v9 = sorted(train_ratings_v7["recipe_id"].unique())

user_to_idx_v9 = {
    user_id: idx
    for idx, user_id in enumerate(user_ids_v9)
}

recipe_to_idx_v9 = {
    recipe_id: idx
    for idx, recipe_id in enumerate(recipe_ids_v9)
}

num_users_v9 = len(user_to_idx_v9)
num_recipes_v9 = len(recipe_to_idx_v9)

print("Users:", num_users_v9)
print("Recipes:", num_recipes_v9)

In [ ]:
import numpy as np

rng = np.random.default_rng(42)

positive_pairs_v9 = set(
    zip(
        positive_train_v9["user_id"],
        positive_train_v9["recipe_id"]
    )
)

all_recipe_ids_v9 = np.array(recipe_ids_v9)

negative_samples_per_positive = 4

train_users_v9 = []
train_recipes_v9 = []
train_labels_v9 = []

for user_id, recipe_id in positive_pairs_v9:

    # Positive
    train_users_v9.append(user_to_idx_v9[user_id])
    train_recipes_v9.append(recipe_to_idx_v9[recipe_id])
    train_labels_v9.append(1.0)

    # Negative samples
    sampled = 0

    while sampled < negative_samples_per_positive:

        negative_recipe_id = rng.choice(all_recipe_ids_v9)

        if (user_id, negative_recipe_id) not in positive_pairs_v9:
            train_users_v9.append(user_to_idx_v9[user_id])
            train_recipes_v9.append(recipe_to_idx_v9[negative_recipe_id])
            train_labels_v9.append(0.0)

            sampled += 1

In [ ]:
train_users_v9 = np.array(train_users_v9)
train_recipes_v9 = np.array(train_recipes_v9)
train_labels_v9 = np.array(train_labels_v9)

print("Training examples:", len(train_labels_v9))
print("Positive:", np.sum(train_labels_v9 == 1))
print("Negative:", np.sum(train_labels_v9 == 0))

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

In [ ]:
embedding_dim = 32

user_input = keras.Input(
    shape=(1,),
    name="user"
)

recipe_input = keras.Input(
    shape=(1,),
    name="recipe"
)

user_embedding = layers.Embedding(
    input_dim=num_users_v9,
    output_dim=embedding_dim,
    embeddings_regularizer=regularizers.l2(1e-6),
    name="user_embedding"
)(user_input)

recipe_embedding = layers.Embedding(
    input_dim=num_recipes_v9,
    output_dim=embedding_dim,
    embeddings_regularizer=regularizers.l2(1e-6),
    name="recipe_embedding"
)(recipe_input)

user_vec = layers.Flatten()(user_embedding)
recipe_vec = layers.Flatten()(recipe_embedding)

# GMF branch
gmf = layers.Multiply()([
    user_vec,
    recipe_vec
])

# MLP branch
mlp = layers.Concatenate()([
    user_vec,
    recipe_vec
])

mlp = layers.Dense(
    64,
    activation="relu",
    kernel_regularizer=regularizers.l2(1e-5)
)(mlp)

mlp = layers.Dropout(0.30)(mlp)

mlp = layers.Dense(
    32,
    activation="relu",
    kernel_regularizer=regularizers.l2(1e-5)
)(mlp)

mlp = layers.Dropout(0.20)(mlp)

combined = layers.Concatenate()([
    gmf,
    mlp
])

output = layers.Dense(
    1,
    activation="sigmoid",
    name="score"
)(combined)

model_v9 = keras.Model(
    inputs=[user_input, recipe_input],
    outputs=output
)

model_v9.summary()

In [ ]:
model_v9.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="binary_crossentropy",
    metrics=[
        keras.metrics.AUC(name="auc")
    ]
)

In [ ]:
early_stopping_v9 = keras.callbacks.EarlyStopping(
    monitor="val_auc",
    mode="max",
    patience=6,
    restore_best_weights=True
)

In [ ]:
history_v9 = model_v9.fit(
    [train_users_v9, train_recipes_v9],
    train_labels_v9,
    validation_split=0.10,
    epochs=40,
    batch_size=1024,
    callbacks=[early_stopping_v9],
    verbose=1
)

In [ ]:
# ==========================================
# V9 - Test data
# ==========================================

test_users_v9 = test_ratings_v7["user_id"].map(user_to_idx_v9)
test_recipes_v9 = test_ratings_v7["recipe_id"].map(recipe_to_idx_v9)

# Uzimamo samo parove koji postoje u mappingu
valid_mask_v9 = (
        test_users_v9.notna()
        & test_recipes_v9.notna()
)

test_users_v9 = test_users_v9[valid_mask_v9].astype(int).values
test_recipes_v9 = test_recipes_v9[valid_mask_v9].astype(int).values

test_labels_v9 = (
    (test_ratings_v7.loc[valid_mask_v9, "rating"] >= 4)
    .astype("float32")
    .values
)

print("Test examples:", len(test_labels_v9))
print("Positive:", np.sum(test_labels_v9 == 1))
print("Negative:", np.sum(test_labels_v9 == 0))

In [ ]:
v9_results = model_v9.evaluate(
    [test_users_v9, test_recipes_v9],
    test_labels_v9,
    return_dict=True
)

print(v9_results)